# Aula 16 — Classes desbalanceadas: amostragem, pesos e avaliação correta

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/joaopaulomirandamatias/ai-lab/blob/main/03-machine-learning/notebooks/16-classes-desbalanceadas-laboratorio.ipynb)

**Objetivo:** comparar intervenções de treino sem alterar a distribuição natural de validação e teste.

Hipóteses pré-registradas:

1. accuracy do baseline constante será alta, apesar de recall positivo zero;
2. pesos e reamostragem aumentarão recall em 0,5, mas poderão reduzir precision e calibração;
3. oversampling aleatório mudará principalmente o prior efetivo; a correção de odds reduzirá Brier/log-loss;
4. ao reduzir a prevalência de avaliação, precision e AP cairão mesmo com ranking semelhante.


## Ambiente e dependências

- Python ≥ 3.10
- NumPy ≥ 1.24
- Matplotlib ≥ 3.7
- scikit-learn ≥ 1.3

Os dados são sintéticos, numéricos e gerados com seed fixa. O SMOTE é implementado de forma didática; em produção, use `imblearn.pipeline.Pipeline`.


In [ ]:
import platform
import warnings

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import sklearn
from sklearn.calibration import calibration_curve
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("error")
SEED = 20260908
rng = np.random.default_rng(SEED)

print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("Matplotlib:", matplotlib.__version__)
print("scikit-learn:", sklearn.__version__)
print("Seed:", SEED)

## 1. População e separação externa

A unidade é um evento independente. Geramos 8.000 eventos com cerca de 2% de positivos e reservamos 20% para validação e 20% para teste. Esses dois conjuntos nunca serão reamostrados.


In [ ]:
X, y = make_classification(
    n_samples=8000,
    n_features=14,
    n_informative=7,
    n_redundant=3,
    n_clusters_per_class=2,
    weights=[0.98, 0.02],
    class_sep=1.25,
    flip_y=0.002,
    random_state=SEED,
)

indices = np.arange(len(y))
idx_train, idx_temp = train_test_split(
    indices, test_size=0.40, stratify=y, random_state=SEED
)
idx_val, idx_test = train_test_split(
    idx_temp, test_size=0.50, stratify=y[idx_temp], random_state=SEED + 1
)
X_train, y_train = X[idx_train], y[idx_train]
X_val, y_val = X[idx_val], y[idx_val]
X_test, y_test = X[idx_test], y[idx_test]

for name, target in [("treino", y_train), ("validação", y_val), ("teste", y_test)]:
    n_pos = int(target.sum())
    print(f"{name:10s}: n={len(target):4d}, positivos={n_pos:3d}, prevalência={target.mean():.6f}")

assert set(idx_train).isdisjoint(idx_val)
assert set(idx_train).isdisjoint(idx_test)
assert set(idx_val).isdisjoint(idx_test)
assert len(idx_train) + len(idx_val) + len(idx_test) == len(y)

## 2. O baseline constante expõe a falha da accuracy

O classificador “sempre negativo” não aprende nada. Ele fornece a referência mínima para decisões binárias.


In [ ]:
pred_constant = np.zeros_like(y_val)
accuracy_constant = accuracy_score(y_val, pred_constant)
balanced_constant = balanced_accuracy_score(y_val, pred_constant)
recall_constant = recall_score(y_val, pred_constant, zero_division=0)

print(f"accuracy:          {accuracy_constant:.6f}")
print(f"balanced accuracy: {balanced_constant:.6f}")
print(f"recall positivo:   {recall_constant:.6f}")
assert accuracy_constant > 0.97
assert balanced_constant == 0.5
assert recall_constant == 0.0

## 3. Preprocessamento ajustado somente no treino

SMOTE depende de distâncias; portanto ajustamos a escala no treino original antes de calcular vizinhos. Validação e teste recebem apenas `transform`.


In [ ]:
scaler = StandardScaler().fit(X_train)
Xs_train = scaler.transform(X_train)
Xs_val = scaler.transform(X_val)
Xs_test = scaler.transform(X_test)

assert np.allclose(Xs_train.mean(axis=0), 0, atol=1e-12)
assert np.isfinite(Xs_val).all() and np.isfinite(Xs_test).all()
print("Shapes:", Xs_train.shape, Xs_val.shape, Xs_test.shape)

## 4. Samplers didáticos

Todas as funções abaixo recebem exclusivamente o treino. Oversampling replica a minoria; undersampling retém três negativos por positivo; SMOTE interpola entre vizinhos minoritários até obter classes balanceadas.


In [ ]:
def random_oversample(X_train_only, y_train_only, generator):
    pos = np.flatnonzero(y_train_only == 1)
    neg = np.flatnonzero(y_train_only == 0)
    extra = generator.choice(pos, size=len(neg) - len(pos), replace=True)
    chosen = np.concatenate([neg, pos, extra])
    chosen = generator.permutation(chosen)
    return X_train_only[chosen], y_train_only[chosen]

def random_undersample(X_train_only, y_train_only, generator, majority_ratio=3):
    pos = np.flatnonzero(y_train_only == 1)
    neg = np.flatnonzero(y_train_only == 0)
    keep_neg = generator.choice(neg, size=min(len(neg), majority_ratio * len(pos)), replace=False)
    chosen = generator.permutation(np.concatenate([keep_neg, pos]))
    return X_train_only[chosen], y_train_only[chosen]

def didactic_smote(X_train_only, y_train_only, generator, k_neighbors=5):
    minority = X_train_only[y_train_only == 1]
    majority = X_train_only[y_train_only == 0]
    neighbors = NearestNeighbors(n_neighbors=k_neighbors + 1).fit(minority)
    neighbor_idx = neighbors.kneighbors(minority, return_distance=False)[:, 1:]
    needed = len(majority) - len(minority)
    base_idx = generator.integers(0, len(minority), size=needed)
    neighbor_choice = generator.integers(0, k_neighbors, size=needed)
    selected_neighbors = neighbor_idx[base_idx, neighbor_choice]
    lam = generator.random((needed, 1))
    synthetic = minority[base_idx] + lam * (minority[selected_neighbors] - minority[base_idx])
    X_out = np.vstack([majority, minority, synthetic])
    y_out = np.concatenate([
        np.zeros(len(majority), dtype=int),
        np.ones(len(minority) + len(synthetic), dtype=int),
    ])
    order = generator.permutation(len(y_out))
    return X_out[order], y_out[order], synthetic

X_over, y_over = random_oversample(Xs_train, y_train, np.random.default_rng(SEED + 10))
X_under, y_under = random_undersample(Xs_train, y_train, np.random.default_rng(SEED + 20))
X_smote, y_smote, X_synthetic = didactic_smote(
    Xs_train, y_train, np.random.default_rng(SEED + 30)
)

for name, target in [("original", y_train), ("oversampling", y_over), ("undersampling", y_under), ("SMOTE", y_smote)]:
    print(f"{name:13s}: n={len(target):5d}, positivos={target.mean():.6f}")

assert y_over.mean() == 0.5 and y_smote.mean() == 0.5
assert np.isclose((y_under == 0).sum() / (y_under == 1).sum(), 3.0)
assert len(X_synthetic) == (y_train == 0).sum() - (y_train == 1).sum()

## 5. Cinco procedimentos pré-especificados

Treinamos regressão logística sem intervenção, com `class_weight="balanced"`, com oversampling, undersampling e SMOTE. Os hiperparâmetros são iguais; apenas a intervenção de classe muda.


In [ ]:
def fit_logistic(X_fit, y_fit, class_weight=None, seed=SEED):
    model = LogisticRegression(
        C=1.0, class_weight=class_weight, max_iter=3000, random_state=seed
    )
    return model.fit(X_fit, y_fit)

models = {
    "sem intervenção": fit_logistic(Xs_train, y_train),
    "pesos balanced": fit_logistic(Xs_train, y_train, class_weight="balanced"),
    "oversampling": fit_logistic(X_over, y_over),
    "undersampling": fit_logistic(X_under, y_under),
    "SMOTE": fit_logistic(X_smote, y_smote),
}

def evaluate(y_true, probabilities, threshold=0.5):
    pred = (probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {
        "accuracy": accuracy_score(y_true, pred),
        "balanced": balanced_accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, probabilities),
        "ap": average_precision_score(y_true, probabilities),
        "brier": brier_score_loss(y_true, probabilities),
        "alerts": int(tp + fp),
        "tp": int(tp), "fp": int(fp), "fn": int(fn), "tn": int(tn),
    }

val_probabilities = {name: model.predict_proba(Xs_val)[:, 1] for name, model in models.items()}
val_metrics = {name: evaluate(y_val, p) for name, p in val_probabilities.items()}

header = f"{'método':17s} {'acc':>7s} {'bal':>7s} {'prec':>7s} {'rec':>7s} {'AP':>7s} {'Brier':>7s} {'alertas':>8s}"
print(header)
for name, m in val_metrics.items():
    print(f"{name:17s} {m['accuracy']:7.3f} {m['balanced']:7.3f} {m['precision']:7.3f} "
          f"{m['recall']:7.3f} {m['ap']:7.3f} {m['brier']:7.3f} {m['alerts']:8d}")

A comparação em 0,5 mostra o efeito sobre a decisão. Ranking e calibração precisam ser lidos em colunas próprias. Pesos e amostragem não garantem ganho de AP; eles mudam o problema de otimização.


In [ ]:
names = list(val_metrics)
x = np.arange(len(names))
width = 0.25
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.bar(x - width, [val_metrics[n]["precision"] for n in names], width, label="precision")
ax.bar(x, [val_metrics[n]["recall"] for n in names], width, label="recall")
ax.bar(x + width, [val_metrics[n]["ap"] for n in names], width, label="AP")
ax.set_xticks(x, names, rotation=18, ha="right")
ax.set_ylim(0, 1)
ax.set_ylabel("Métrica")
ax.set_title("Validação natural: a intervenção altera os trade-offs")
ax.legend()
fig.tight_layout()
plt.show()

## 6. Avaliação confirmatória no teste natural

Os cinco procedimentos foram definidos antes deste acesso ao teste. Executamos uma comparação confirmatória única e não escolhemos novos hiperparâmetros a partir dos resultados.


In [ ]:
test_probabilities = {name: model.predict_proba(Xs_test)[:, 1] for name, model in models.items()}
test_metrics = {name: evaluate(y_test, p) for name, p in test_probabilities.items()}

print(header)
for name, m in test_metrics.items():
    print(f"{name:17s} {m['accuracy']:7.3f} {m['balanced']:7.3f} {m['precision']:7.3f} "
          f"{m['recall']:7.3f} {m['ap']:7.3f} {m['brier']:7.3f} {m['alerts']:8d}")

assert test_metrics["sem intervenção"]["accuracy"] > 0.97
assert test_metrics["pesos balanced"]["recall"] > test_metrics["sem intervenção"]["recall"]
assert test_metrics["pesos balanced"]["balanced"] > test_metrics["sem intervenção"]["balanced"]
assert test_metrics["pesos balanced"]["alerts"] > test_metrics["sem intervenção"]["alerts"]

## 7. Corrigindo o prior do oversampling

O oversampling deixou o treino efetivamente 50/50. Sob a hipótese de prior shift, ajustamos as odds para a prevalência natural do treino. A transformação é monotônica: o ranking permanece, mas as probabilidades mudam.


In [ ]:
def adjust_prior(probabilities, sampled_prior, target_prior, eps=1e-10):
    q = np.clip(probabilities, eps, 1 - eps)
    odds_sampled = q / (1 - q)
    prior_ratio = (target_prior / (1 - target_prior)) / (
        sampled_prior / (1 - sampled_prior)
    )
    odds_target = odds_sampled * prior_ratio
    return odds_target / (1 + odds_target)

p_over_raw = test_probabilities["oversampling"]
p_over_corrected = adjust_prior(p_over_raw, y_over.mean(), y_train.mean())

raw_scores = {
    "brier": brier_score_loss(y_test, p_over_raw),
    "log_loss": log_loss(y_test, p_over_raw),
    "roc_auc": roc_auc_score(y_test, p_over_raw),
}
corrected_scores = {
    "brier": brier_score_loss(y_test, p_over_corrected),
    "log_loss": log_loss(y_test, p_over_corrected),
    "roc_auc": roc_auc_score(y_test, p_over_corrected),
}
print("oversampling bruto:   ", {k: round(v, 6) for k, v in raw_scores.items()})
print("prior corrigido:      ", {k: round(v, 6) for k, v in corrected_scores.items()})

assert np.isclose(raw_scores["roc_auc"], corrected_scores["roc_auc"])
assert corrected_scores["brier"] < raw_scores["brier"]
assert corrected_scores["log_loss"] < raw_scores["log_loss"]

### Diagrama de confiabilidade

Bins por quantis distribuem aproximadamente a mesma quantidade de casos por ponto. Ainda assim, poucos positivos implicam incerteza; o gráfico é diagnóstico, não prova isolada.


In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5))
for label, p in [("oversampling bruto", p_over_raw), ("prior corrigido", p_over_corrected)]:
    observed, predicted = calibration_curve(y_test, p, n_bins=8, strategy="quantile")
    ax.plot(predicted, observed, marker="o", label=label)
ax.plot([0, 1], [0, 1], "--", color="black", label="ideal")
ax.set(xlabel="Probabilidade média prevista", ylabel="Frequência positiva", title="Efeito da correção de prior")
ax.legend()
fig.tight_layout()
plt.show()

## 8. Mesma regra, prevalências diferentes

Escolhemos na validação um threshold para o modelo sem intervenção com recall de pelo menos 70% e a maior precision possível. Depois reamostramos apenas para criar populações de avaliação hipotéticas; o modelo e o threshold ficam congelados.


In [ ]:
candidate_thresholds = np.unique(val_probabilities["sem intervenção"])
feasible = []
for threshold in candidate_thresholds:
    m = evaluate(y_val, val_probabilities["sem intervenção"], threshold)
    if m["recall"] >= 0.70:
        feasible.append((m["precision"], threshold))
assert feasible
fixed_threshold = float(max(feasible)[1])
print(f"threshold congelado: {fixed_threshold:.6f}")

def resample_with_prevalence(y_true, p, prevalence, n, generator):
    n_pos = int(round(prevalence * n))
    n_neg = n - n_pos
    pos_idx = generator.choice(np.flatnonzero(y_true == 1), size=n_pos, replace=True)
    neg_idx = generator.choice(np.flatnonzero(y_true == 0), size=n_neg, replace=True)
    chosen = generator.permutation(np.concatenate([pos_idx, neg_idx]))
    return y_true[chosen], p[chosen]

shift_results = []
for j, prevalence in enumerate([0.005, 0.02, 0.10]):
    ys, ps = resample_with_prevalence(
        y_test, test_probabilities["sem intervenção"], prevalence, 20000,
        np.random.default_rng(SEED + 100 + j),
    )
    m = evaluate(ys, ps, fixed_threshold)
    shift_results.append((prevalence, m))
    print(
        f"prevalência={prevalence:.3f} | ROC-AUC={m['roc_auc']:.4f} | "
        f"AP={m['ap']:.4f} | precision={m['precision']:.4f} | recall={m['recall']:.4f}"
    )

assert shift_results[0][1]["precision"] < shift_results[-1][1]["precision"]
assert shift_results[0][1]["ap"] < shift_results[-1][1]["ap"]

## 9. Verificações finais

Estas verificações garantem separação mecânica e confirmam as hipóteses. Elas não demonstram validade externa nem tornam pontos sintéticos equivalentes a novos casos reais.


In [ ]:
assert len(set(idx_train) | set(idx_val) | set(idx_test)) == len(y)
assert y_val.mean() < 0.03 and y_test.mean() < 0.03
assert y_over.mean() == 0.5 and y_smote.mean() == 0.5
assert np.isfinite(X_synthetic).all()
assert all(np.isfinite(p).all() for p in test_probabilities.values())
assert accuracy_constant > 0.97 and recall_constant == 0
assert corrected_scores["brier"] < raw_scores["brier"]
assert abs(raw_scores["roc_auc"] - corrected_scores["roc_auc"]) < 1e-12

print("Todas as verificações passaram.")
print(
    f"Baseline constante: accuracy={accuracy_constant:.6f}, "
    f"balanced accuracy={balanced_constant:.6f}, recall={recall_constant:.6f}."
)
print(
    f"Teste — sem intervenção: AP={test_metrics['sem intervenção']['ap']:.6f}, "
    f"recall={test_metrics['sem intervenção']['recall']:.6f}; "
    f"pesos: recall={test_metrics['pesos balanced']['recall']:.6f}."
)
print(
    f"Oversampling: Brier {raw_scores['brier']:.6f} → "
    f"{corrected_scores['brier']:.6f} após correção de prior."
)

## Conclusão

- A accuracy do classificador constante confirmou por que taxa de acerto isolada não basta.
- Pesos e reamostragem alteraram o trade-off no threshold 0,5; não criaram informação nova.
- A avaliação natural mostrou o volume de alertas e a precision que seriam ocultados por um teste balanceado.
- A correção de prior melhorou os proper scores sem alterar o ranking, como previsto.
- Ao mudar apenas a prevalência da avaliação, AP e precision mudaram de modo expressivo.

**Desafio:** repita o experimento dentro de `imblearn.pipeline.Pipeline` com `StratifiedKFold`. Verifique programaticamente que cada sampler recebe apenas os índices de treino do fold — ponte para a Aula 17.
